In [ ]:
# %%
from xc import XenoCantoDownloader
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

xcd = XenoCantoDownloader(api_key=os.environ["XC_API_KEY"])


import librosa
# %%
import json

with open("../data/san_diego_xc_aux/xc_meta_aux.json", mode="r") as f:
    data = json.load(f)
    # json.dump(data, f, indent=4)

# %%
import requests

# %%
import shutil
import os
from pathlib import Path
from multiprocessing.pool import ThreadPool
import tqdm

# # https://stackoverflow.com/questions/16694907/download-large-file-in-python-with-requests
# def download_file(url, local_filename, dry_run=False):
#     if os.path.exists(local_filename):
#         try:
#             librosa.load(path=local_filename)
#             return local_filename
#         except Exception as e:
#             pass
        
#     try:
#         with requests.get(url, stream=True) as r:
#             with open(local_filename, 'wb') as f:
#                 if not dry_run:
#                     shutil.copyfileobj(r.raw, f)
#                 else:
#                     print(local_filename)

#         return local_filename
#     except Exception as e:
#         print(e, flush=True)
#         return None


def prep_download(args, dry_run=False):
        url = args[0]
        local_filename = args[1]
        if os.path.exists(local_filename):
            try:
                librosa.load(path=local_filename)
                return local_filename
            except Exception as e:
                print(local_filename, e, "bad file, remake")

        try:
            with requests.get(url, stream=True) as r:
                with open(local_filename, 'wb') as f:
                    if not dry_run:
                        shutil.copyfileobj(r.raw, f)
                    else:
                        print(local_filename)

            return local_filename
        except Exception as e:
            print(local_filename, e, flush=True)
            return None

def download_files(xcd, data, parent_folder="../data/san_diego_xc_aux/xeno-canto", workers = 2):
    

    os.makedirs(parent_folder, exist_ok=True)

    if "recordings" in data[0]:
        data = xcd.concat_recording_data(data) 
    download_data = [
        (recording["file"], Path(parent_folder) / Path(recording["file-name"].replace("/", "_")))
        for recording in data
    ]

    with ThreadPool(processes=1024) as pool:
        print("Main process: Submitting tasks...")
        
        # Iterate over the results to wait for all tasks to complete.
        # This loop will block until all tasks are finished.
        for result in tqdm.tqdm(pool.imap_unordered(prep_download, download_data), total=len(download_data)):
            if result is None:
                print("ISSUE")
    
    return results

results = download_files(xcd, data)
results

# %%
import pandas as pd
recordings = xcd.concat_recording_data(data)
df = pd.DataFrame(recordings)

df.shape

# %%





In [ ]:
132510 / 303

In [ ]:
df["en"].value_counts()[df["en"].value_counts() < 1000].hist(bins=50)

In [ ]:
df["grp"].value_counts()